In [0]:
import csv
import io
import re
from datetime import datetime, timedelta, timezone
from uuid import uuid4

In [0]:
VOLUME = "/Volumes/workspace/e-commerce/e-commerce_sales_analysis"
SEED = f"{VOLUME}/seed/global_ecommerce_sales.csv"
RAW = f"{VOLUME}/raw_data"
ROWS_PER_RUN = 100
HEADER = ["Order_ID", "Country", "Category", "Unit_Price", "Quantity",
          "Order_Date", "Total_Amount"]
SNAPSHOT = re.compile(
    r"global_ecommerce_sales_\d{8}T\d{6}\+0700_(\d+)_(\w+)\.csv"
)

In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id").strip() or uuid4().hex
if not re.fullmatch(r"[A-Za-z0-9]+", run_id):
    raise ValueError("Set run_id to {{job.run_id}} in the notebook task.")

In [0]:
def publish_snapshot():
    dbutils.fs.mkdirs(RAW)
    published_rows = 0
    for file in dbutils.fs.ls(RAW):
        match = SNAPSHOT.fullmatch(file.name)
        if match:
            if match[2] == run_id:
                print(f"Run already published {file.path}")
                return
            published_rows = max(published_rows, int(match[1]))

    # The seed is a small demo CSV. Check the size before using fs.head.
    seed_info = next(
        (file for file in dbutils.fs.ls(f"{VOLUME}/seed")
         if file.name == "global_ecommerce_sales.csv"),
        None,
    )
    if seed_info is None or not 0 < seed_info.size <= 1_000_000:
        raise ValueError(f"Upload the demo CSV (up to 1 MB) to {SEED}")
    reader = csv.DictReader(io.StringIO(dbutils.fs.head(SEED, seed_info.size + 1)))
    if reader.fieldnames != HEADER:
        raise ValueError(f"Unexpected seed header: {reader.fieldnames}")
    rows = sorted(reader, key=lambda row: int(row["Order_ID"]))
    if not rows or len({int(row["Order_ID"]) for row in rows}) != len(rows):
        raise ValueError("The seed must contain unique Order_ID values.")

    if published_rows >= len(rows):
        print(f"Simulation complete: {len(rows)} orders already published.")
        return

    count = min(published_rows + ROWS_PER_RUN, len(rows))
    payload = io.StringIO(newline="")
    writer = csv.DictWriter(payload, fieldnames=HEADER, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows[:count])
    bangkok = timezone(timedelta(hours=7))
    timestamp = datetime.now(bangkok).strftime("%Y%m%dT%H%M%S%z")
    path = f"{RAW}/global_ecommerce_sales_{timestamp}_{count:06d}_{run_id}.csv"
    if not dbutils.fs.put(path, payload.getvalue(), overwrite=False):
        raise IOError(f"Could not publish {path}")
    print(f"Published {count} cumulative orders to {path}")

In [0]:
publish_snapshot()